# TensorEE quickstart: a trained water U-Net on one Sentinel-1 chip

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noelEOS/tensor-ee/blob/main/notebooks/01_quickstart.ipynb)

This notebook runs a trained neural network inside Google Earth Engine. The model is a U-Net
with a ResNet-18 encoder that maps open water in Sentinel-1 radar imagery. It takes one chip of
224 × 224 pixels at 10 m (2.24 km on a side) and returns, for every pixel, the probability
that it is water and a water mask.

Earth Engine computes the whole prediction from your Sentinel-1 scene. Nothing is downloaded
and no model server is involved: the network's arithmetic is a large Earth Engine computation,
and its weights are public Earth Engine table assets. You do not need the model's checkpoint
file.

**What you need.** Nothing, to read and run the notebook offline. To run the model, an Earth
Engine account and a Cloud project
[registered for Earth Engine](https://developers.google.com/earth-engine/guides/access) where
you can create an image asset.

**What it costs.** Offline, nothing. With `RUN_LIVE = True` the notebook makes a few small
requests (to place the chip and find a scene) and **one batch export of one chip**, billed to
your project. The first recorded export of this chip took 140 s and 64.9 batch EECU-seconds.
The export creates one image asset in your project (about 1 MB). Each step says what it does
before it does it.

## 1. Install

The trained U-Net's Earth Engine computation is stored in the repository, in the file the
[Code Editor demo](https://github.com/noelEOS/tensor-ee/blob/main/CODE_EDITOR.md) uses. That
file is not part of the package on PyPI, so this cell copies the repository and installs
TensorEE from it. It also installs `folium` for the map. In Colab it takes a minute or two.

If you opened this notebook from your own copy of the repository, you can skip this cell.

In [ ]:
# The Code Editor module (demos/code_editor/segwater.js) is in the repository, not on PyPI,
# so this notebook installs TensorEE from a copy of the repository.
!git clone --depth 1 https://github.com/noelEOS/tensor-ee.git
%pip install --quiet ./tensor-ee folium

In [ ]:
import sys
from pathlib import Path


def find_repository():
    """The copy of the repository: ./tensor-ee in Colab, or the folder this notebook is in."""
    for folder in (Path("tensor-ee"), Path("."), Path("..")):
        if (folder / "demos" / "code_editor" / "segwater.js").is_file():
            return folder.resolve()
    raise FileNotFoundError("Run the install cell first: it copies the repository to ./tensor-ee")


REPOSITORY = find_repository()
sys.path.insert(0, str(REPOSITORY))
print("Using the repository in", REPOSITORY)

## 2. Settings

This is the only cell you need to edit.

- `PROJECT`: your Cloud project ID, the one you use in the Code Editor.
- `RUN_LIVE`: leave it `False` to run everything offline first. Set it to `True` to sign in
  and run the steps that use your project.
- `LONGITUDE`, `LATITUDE`: the centre of the chip, in degrees. The default is Sun Moon Lake,
  Taiwan, where the model's Earth Engine result was checked against PyTorch.
- `START_DATE`, `END_DATE`: the period in which to look for a Sentinel-1 scene.
- `ASSET_NAME`: the name of the new image asset the export creates in your project. Use a new
  name for each run: an export never overwrites an asset.

After changing `RUN_LIVE`, restart the runtime (in Colab: **Runtime → Restart session**) and
run the notebook again from the top: the offline mode stays installed until the runtime
restarts.

In [ ]:
PROJECT = "YOUR_EE_PROJECT"   # your Cloud project registered for Earth Engine
RUN_LIVE = False              # True: sign in and run the steps that use your project

LONGITUDE, LATITUDE = 120.9150, 23.8620          # Sun Moon Lake, Taiwan
START_DATE, END_DATE = "2024-01-01", "2024-01-08"  # look for Sentinel-1 scenes in this period
ASSET_NAME = "tensor_ee_quickstart_chip"         # a new image asset in your project

In [ ]:
PLACEHOLDER = "YOUR_EE_PROJECT"


def live_step(what):
    """True when a step that uses Earth Engine should run; it refuses a placeholder project."""
    if not RUN_LIVE:
        print(f"Skipped: {what}. RUN_LIVE is False, so nothing was sent to Earth Engine.")
        return False
    if not PROJECT or PROJECT == PLACEHOLDER:
        raise ValueError("Set PROJECT to your own Cloud project ID in the settings cell "
                         "before running with RUN_LIVE = True.")
    return True

## 3. Sign in, or work offline

With `RUN_LIVE = True` the next cell signs you in. The first time, `ee.Authenticate()` shows
a link or a prompt; follow it with the Google account you use for Earth Engine. Every request
after that is made with your account and billed to `PROJECT`.

With `RUN_LIVE = False`, the cell after it installs the Earth Engine client's offline
function signatures (`ee.apitestcase.ApiTestCase`, a test helper that ships with the client).
They let the notebook build the same Earth Engine computation without credentials and without
sending anything.

In [ ]:
import ee

if live_step("sign in to Earth Engine"):
    ee.Authenticate()
    ee.Initialize(project=PROJECT)
    print("Signed in. Requests are billed to", PROJECT)

In [ ]:
import ee

if not RUN_LIVE:
    from ee.apitestcase import ApiTestCase

    offline = ApiTestCase()
    offline.setUp()   # offline signatures: no credentials, no network
    print("Offline: the notebook builds Earth Engine computations but sends none.")

## 4. Load the trained model

The model comes as one file, `demos/code_editor/segwater.js`. TensorEE built it from the
trained checkpoint: it holds the complete Earth Engine computation for one chip, with blanks
for the place and the scene, and it reads the network's weights from 29 public Earth Engine
table assets. That is why you need neither the checkpoint nor an upload. The same file runs in
the Code Editor demo.

The file stores the computation compressed. Reading it checks that it is complete and
undamaged.

In [ ]:
from demos.code_editor import build as segwater_module

TEMPLATE, METADATA = segwater_module.parse_module(
    (REPOSITORY / "demos" / "code_editor" / "segwater.js").read_text())

print("Model computation:", f"{METADATA['templateBytes']:,}", "characters, checked intact")
print("Weight tables it reads:", len(METADATA["weightAssets"]))
print("Built from the checkpoint with SHA-256", METADATA["checkpointSha256"][:16] + "...")

## 5. Choose the place

The chip is a 224 × 224 grid of 10 m pixels in the local UTM zone, centred on your point.
The model was trained on chips of exactly this size, so the notebook keeps it.

Offline, the notebook uses the preset chip at Sun Moon Lake. With `RUN_LIVE = True` the live
cell asks Earth Engine to convert your longitude and latitude to UTM metres: **one small
request**.

In [ ]:
from tensor_ee import Grid

CHIP, PIXEL = 224, 10          # pixels per side, metres per pixel
PRESET = {                     # the verified Sun Moon Lake chip of the Code Editor demo
    "crs": "EPSG:32651", "x": 286030, "y": 2640860,
    "scene": "COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20240103T100134_20240103T100159_051941_06469E_B0BD",
}


def utm_zone(longitude, latitude):
    """The EPSG code of the WGS 84 UTM zone containing a point."""
    zone = min(60, max(1, int((longitude + 180) // 6) + 1))
    return f"EPSG:{326 if latitude >= 0 else 327}{zone:02d}"


place = {"crs": PRESET["crs"], "x": PRESET["x"], "y": PRESET["y"]}   # offline: the preset

In [ ]:
if live_step("place the chip at your point (1 request)"):
    crs = utm_zone(LONGITUDE, LATITUDE)
    x, y = ee.Geometry.Point([LONGITUDE, LATITUDE]).transform(crs, 0.01).coordinates().getInfo()
    half = CHIP * PIXEL // 2
    # The chip's upper-left corner, on whole 10 m steps.
    place = {"crs": crs, "x": round((x - half) / PIXEL) * PIXEL,
             "y": round((y + half) / PIXEL) * PIXEL}

In [ ]:
grid = Grid(place["crs"], (PIXEL, 0, place["x"], 0, -PIXEL, place["y"]), CHIP, CHIP)
print("Chip grid:", grid.crs, "upper-left corner", (place["x"], place["y"]),
      f"{CHIP} x {CHIP} pixels of {PIXEL} m")

## 6. Find a Sentinel-1 scene

The model reads the VV and VH bands of a Sentinel-1 GRD scene (interferometric wide swath, 10 m)
in decibels, as Earth Engine stores them. The scene must cover the whole chip.

With `RUN_LIVE = True` the live cell lists up to ten scenes in your period and, for each, the
fraction of the chip it covers, and picks the first that covers it all: **one request**, which
reads only the chip's area. Offline, the notebook uses the preset scene of 3 January 2024.

In [ ]:
scene_id = PRESET["scene"]   # offline: the preset scene

In [ ]:
if live_step("find a Sentinel-1 scene (1 request)"):
    area = grid.geometry()
    collection = (ee.ImageCollection("COPERNICUS/S1_GRD")
                  .filterBounds(area).filterDate(START_DATE, END_DATE)
                  .filter(ee.Filter.eq("instrumentMode", "IW"))
                  .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
                  .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
                  .filter(ee.Filter.eq("resolution_meters", 10))
                  .sort("system:time_start"))

    def describe(image):
        image = ee.Image(image)
        valid = (image.select(["VV", "VH"]).mask().reduce(ee.Reducer.min()).gt(0)
                 .unmask(0, False).rename("valid"))
        coverage = valid.reduceRegion(reducer=ee.Reducer.mean(), geometry=area, crs=grid.crs,
                                      crsTransform=list(grid.transform),
                                      maxPixels=CHIP * CHIP, tileScale=2).get("valid")
        return ee.Dictionary({
            "id": ee.String("COPERNICUS/S1_GRD/").cat(image.get("system:index")),
            "time": image.date().format("YYYY-MM-dd HH:mm"),
            "pass": image.get("orbitProperties_pass"), "coverage": coverage})

    scenes = collection.toList(10).map(describe).getInfo()
    for scene in scenes:
        print(f"{scene['time']}  {scene['pass']:<10}  covers {100 * (scene['coverage'] or 0):6.2f}%  {scene['id']}")
    complete = [scene for scene in scenes if (scene["coverage"] or 0) >= 0.999]
    if not complete:
        raise RuntimeError("No scene in this period covers the whole chip. Widen START_DATE to "
                           "END_DATE, or move the point away from the edge of the swath.")
    scene_id = complete[0]["id"]
    print("Using", scene_id)

## 7. Build the prediction

This cell fills the model's blanks with your grid and scene and turns the text back into an
Earth Engine image. Nothing is computed yet: like any Earth Engine image, it is a recipe that
Earth Engine evaluates only when you export or display it.

The recipe converts the radar values from decibels the way the model was trained, runs the
U-Net, and returns five bands:

| Band | Meaning |
|---|---|
| `logit_0`, `logit_1` | the network's raw scores for "not water" and "water" |
| `water_probability` | the probability of water, from 0 to 1 |
| `water_mask` | 1 where the probability is at least 0.5, else 0 |
| `coverage` | 1 where both radar bands and the result are valid |

For the preset chip, the notebook also checks the computation against the record: it should
be identical, byte for byte, to the one TensorEE's own Python recipe builds with the trained
checkpoint, which the repository records.

In [ ]:
import json

sys.setrecursionlimit(max(sys.getrecursionlimit(), 20000))   # the U-Net is a deep graph
OUTPUT_BANDS = list(segwater_module.OUTPUT_BANDS)

text = segwater_module.substitute(TEMPLATE, grid.crs, scene_id,
                                  (place["x"], place["y"]), (place["x"], place["y"]))
chip = ee.Image(ee.deserializer.fromJSON(text))
# Put the chip on its grid, as the library's recipe does for every chip it computes.
prediction = (ee.ImageCollection.fromImages([chip]).mosaic().select(OUTPUT_BANDS)
              .setDefaultProjection(grid.projection()).clip(grid.geometry())
              .updateMask(grid.mask()).toFloat())

serialized = prediction.serialize()
graph_sha256 = segwater_module.sha256(serialized)
print("Bands:", OUTPUT_BANDS)
print(f"The computation is {len(serialized):,} characters of Earth Engine expression.")

record = json.loads((REPOSITORY / "docs" / "evidence" / "s03-template-identity.json").read_text())
preset = next(aoi for aoi in record["aois"] if aoi["name"] == "segwater_sun_moon_lake_1x1")
if (scene_id, grid.crs, place["x"], place["y"]) != (PRESET["scene"], PRESET["crs"], PRESET["x"], PRESET["y"]):
    print("Your own place: there is no recorded computation to compare it with.")
elif ee.__version__ != METADATA["earthengineApiVersion"]:
    print(f"Not compared: your Earth Engine client is {ee.__version__}; the record was made with "
          f"{METADATA['earthengineApiVersion']}, and client versions write expressions differently.")
elif graph_sha256 == preset["library_sha256"]:
    print("Identical, byte for byte, to the recorded computation of the library's recipe.")
else:
    print("Different from the recorded computation: please report this as an issue.")

## 8. Export the prediction (billed to your project)

**What this creates.** One batch export task, `Export.image.toAsset`, and when it completes,
one new image asset, `projects/PROJECT/assets/ASSET_NAME`: five float32 bands over
224 × 224 pixels, about 1 MB.

**What it costs.** The first recorded export of the Sun Moon Lake chip took 140 s and
64.9 batch EECU-seconds. No cost has been recorded for other places. How long an export
waits and runs varies with Earth Engine's load.

The model runs as an export, not as map tiles: when a whole trained U-Net was sent as one
interactive request, it exceeded Earth Engine's memory limit. The cell after this one waits for the task.

In [ ]:
asset_id = f"projects/{PROJECT}/assets/{ASSET_NAME}"

if live_step("export the prediction (1 batch task, 1 new asset)"):
    payload = prediction.set({"source_scene": scene_id,
                              "checkpoint_sha256": METADATA["checkpointSha256"],
                              "template_sha256": METADATA["templateSha256"]})
    task = ee.batch.Export.image.toAsset(
        image=payload, description="tensor_ee_quickstart", assetId=asset_id,
        region=grid.geometry(), crs=grid.crs, crsTransform=list(grid.transform),
        maxPixels=CHIP * CHIP * len(OUTPUT_BANDS), pyramidingPolicy={".default": "sample"})
    task.start()
    print("Started task", task.id, "writing", asset_id)

Waiting checks the task's status every 20 seconds. You can also follow it in the **Tasks** tab
of the Code Editor. If you stop waiting, the export carries on; run this cell again later.

In [ ]:
import time

if live_step("wait for the export"):
    deadline = time.time() + 30 * 60
    while True:
        state = task.status()["state"]
        if state in ("COMPLETED", "FAILED", "CANCELLED"):
            break
        if time.time() > deadline:
            raise TimeoutError("Still running after 30 minutes; run this cell again later.")
        time.sleep(20)
    print("Export", state)
    if state != "COMPLETED":
        raise RuntimeError(task.status().get("error_message", "the export did not complete"))

## 9. See the radar image and the water map

The map shows the Sentinel-1 VV band and the completed asset's water probability and mask.
The layers are Earth Engine map tiles: displaying them reads the saved asset and the scene,
so panning does not run the network again. Each tile you view is a small interactive request.
Use the layer control at the top right to switch layers.

In [ ]:
if live_step("show the result on a map"):
    import folium
    from IPython.display import display

    def tile_layer(image, visualization, name, show=True):
        url = ee.Image(image).getMapId(visualization)["tile_fetcher"].url_format
        return folium.raster_layers.TileLayer(
            tiles=url, name=name, overlay=True, show=show,
            attr="Google Earth Engine; contains modified Copernicus Sentinel data")

    result = ee.Image(asset_id)
    outline = grid.geometry().transform("EPSG:4326", 0.01).getInfo()
    longitudes, latitudes = zip(*outline["coordinates"][0])
    water_map = folium.Map(location=[sum(latitudes) / len(latitudes), sum(longitudes) / len(longitudes)],
                           zoom_start=14)
    tile_layer(ee.Image(scene_id).select("VV"), {"min": -25, "max": 0},
               "Sentinel-1 VV (dB)").add_to(water_map)
    tile_layer(result.select("water_probability"),
               {"min": 0, "max": 1, "palette": ["ffffff", "08306b"]},
               "Water probability").add_to(water_map)
    tile_layer(result.select("water_mask").selfMask(), {"palette": ["00bfff"]},
               "Water mask", show=False).add_to(water_map)
    folium.GeoJson(outline, name="Chip", style_function=lambda _: {
        "fill": False, "color": "#ff7f00", "weight": 2}).add_to(water_map)
    folium.LayerControl().add_to(water_map)
    display(water_map)

## What this shows, and what it does not

- **The computation.** For the preset chip, the notebook's computation is identical to the
  library's own recipe built with the trained checkpoint (step 7). For another place it is the
  same computation with a different grid and scene.
- **Earth Engine against PyTorch.** For the preset chip, an earlier export was compared pixel by
  pixel with the trained model in PyTorch: the largest logit difference was 1.35e-5, the largest
  probability difference 6.4e-6, and the water masks agreed on all 50,176 pixels
  ([validation](https://github.com/noelEOS/tensor-ee/blob/main/docs/validation.md)). Your
  export at another place has not been compared; that needs the checkpoint, which is not yet
  public.
- **Accuracy.** The water map is a prediction, not ground truth, and this notebook does not
  measure how often it is right. The model was trained on Sentinel-1 chips with water labels
  derived from Sentinel-2
  ([model notes](https://github.com/noelEOS/tensor-ee/blob/main/docs/model.md)).
- **Size.** One chip of 224 × 224 pixels per export. The Code Editor demo runs up to four chips
  side by side; each chip is computed on its own, so seams can appear at chip borders.

**Cleaning up.** The asset stays in your project until you delete it, in the Code Editor's
**Assets** tab.

**Next.** [Notebook 02](https://github.com/noelEOS/tensor-ee/blob/main/notebooks/02_is_my_model_supported.ipynb)
checks whether your own model can run this way, and
[notebook 03](https://github.com/noelEOS/tensor-ee/blob/main/notebooks/03_bring_your_own_cnn.ipynb)
takes a trained U-Net of your own to Earth Engine.